In [1]:
# You may need to install these
# pip install pooch requests

import requests
import json

In [2]:
import json
import requests
# Here's a sample JSON structure similar to what APIs return
sample_json = '''
{
  "station": "USC00305800",
  "name": "New York Central Park",
  "location": {
    "latitude": 40.7789,
    "longitude": -73.9692
  },
  "observations": [
    {"date": "2023-01-01", "temperature": 32, "precipitation": 0.0},
    {"date": "2023-01-02", "temperature": 28, "precipitation": 0.5},
    {"date": "2023-01-03", "temperature": 35, "precipitation": 0.0},
    {"date": "2023-01-04", "temperature": 38, "precipitation": 0.2},
    {"date": "2023-01-05", "temperature": 41, "precipitation": 0.0}
  ]
}
'''

# Parse the JSON
data = json.loads(sample_json)

# Access nested data
print("Station:", data['station'])
print("Location:", data['location'])
print("First observation:", data['observations'][0])

Station: USC00305800
Location: {'latitude': 40.7789, 'longitude': -73.9692}
First observation: {'date': '2023-01-01', 'temperature': 32, 'precipitation': 0.0}


In [3]:
# 1. Extract and print all dates and temperatures (8 points)
print("Date, Temperature")
for obs in data['observations']:
    print(obs['date'], obs['temperature'])

Date, Temperature
2023-01-01 32
2023-01-02 28
2023-01-03 35
2023-01-04 38
2023-01-05 41


In [4]:
# 2. Calculate average temperature (8 points)
total_temp = 0
count = 0
for obs in data['observations']:
    total_temp += obs['temperature']
    count += 1

avg_temp = total_temp / count
print(f"Average temperature: {avg_temp}°F")

Average temperature: 34.8°F


In [5]:
# 3. Find days with precipitation (9 points)
print("\nDays with precipitation:")
for obs in data['observations']:
    if obs['precipitation'] > 0:
        print(obs['date'], obs['precipitation'])


Days with precipitation:
2023-01-02 0.5
2023-01-04 0.2


In [6]:
lat, lon = 40.7789, -73.9692
headers = {"User-Agent": "Climate-homework (wt2394@columbia.edu)"}

points_url = f"https://api.weather.gov/points/{lat},{lon}"
r1 = requests.get(points_url, headers=headers)
r1.raise_for_status()
points_data = r1.json()

forecast_url = points_data["properties"]["forecast"]
r2 = requests.get(forecast_url, headers=headers)
r2.raise_for_status()
forecast_data = r2.json()

periods = forecast_data["properties"]["periods"]

print("Forecast (first 6 periods):")
for p in periods[:6]:
    name = p["name"]
    temp = p["temperature"]
    unit = p["temperatureUnit"]
    short = p["shortForecast"]
    print(f"{name}: {temp}{unit} - {short}")


Forecast (first 6 periods):
Tonight: 32F - Rain And Snow Likely
Washington's Birthday: 39F - Slight Chance Light Snow then Mostly Cloudy
Monday Night: 31F - Mostly Cloudy then Slight Chance Snow Showers
Tuesday: 45F - Slight Chance Snow Showers then Partly Sunny
Tuesday Night: 36F - Mostly Cloudy
Wednesday: 43F - Light Rain Likely


In [8]:
%pip install pooch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
import pooch

# Set up Pooch to download a file
# This example downloads a small air quality dataset
file_path = pooch.retrieve(
    url="https://github.com/pandas-dev/pandas/raw/main/doc/data/air_quality_no2.csv",
    known_hash=None
)

print("File downloaded to:", file_path)
print("File exists:", os.path.exists(file_path))

File downloaded to: /home/wt2394/.cache/pooch/458dad453f6a48e510cd544bef1854e3-air_quality_no2.csv
File exists: True


In [10]:
file_size = os.path.getsize(file_path)
print(f"File size: {file_size} bytes")

line_count = 0
with open(file_path, "r") as f:
    for _ in f:
        line_count += 1

print(f"Number of lines: {line_count}")

File size: 31984 bytes
Number of lines: 1036


In [11]:
my_url = "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"
my_file = pooch.retrieve(url=my_url, known_hash=None)

print("My file downloaded to:", my_file)
print("My file exists:", os.path.exists(my_file))
print("My file size:", os.path.getsize(my_file), "bytes")
print("3. 091fa6f46c6d0e4f56f340e4282e9175-GLB.Ts+dSST.csv – NASA GISS global temperature anomalies (Land–Ocean)")

My file downloaded to: /home/wt2394/.cache/pooch/091fa6f46c6d0e4f56f340e4282e9175-GLB.Ts+dSST.csv
My file exists: True
My file size: 12878 bytes
3. 091fa6f46c6d0e4f56f340e4282e9175-GLB.Ts+dSST.csv – NASA GISS global temperature anomalies (Land–Ocean)


In [12]:
print("\nData Inventory:")
print("1. meteorites.csv – NASA meteorite landings")
print("2. air_quality_no2.csv – Air quality NO2 measurements")
print("3. GLB.Ts+dSST.csv – NASA GISS global temperature anomalies")


Data Inventory:
1. meteorites.csv – NASA meteorite landings
2. air_quality_no2.csv – Air quality NO2 measurements
3. GLB.Ts+dSST.csv – NASA GISS global temperature anomalies


In [13]:
#Part3
import requests

# OPeNDAP provides metadata in different formats
# We'll get basic info about a climate dataset

base_url = "http://iridl.ldeo.columbia.edu/expert/SOURCES/.NOAA/.NCEP/.CPC/.UNIFIED_PRCP/.GAUGE_BASED/.GLOBAL/.v1p0/.Monthly/.RETRO/.rain/dods"

# Get DDS (Dataset Descriptor Structure) - describes the structure
dds_url = base_url + ".dds"
response = requests.get(dds_url)

print("Dataset Structure:")
print(response.text[:500])  # Print first 500 characters

Dataset Structure:
Dataset {
    Float32 T[T = 324];
    Float32 Y[Y = 360];
    Float32 X[X = 720];
    Grid {
     ARRAY:
        Float32 rain[T = 324][Y = 360][X = 720];
     MAPS:
        Float32 T[T = 324];
        Float32 Y[Y = 360];
        Float32 X[X = 720];
    } rain;
} rain;



Task1 
Dimension names: T, Y, X
Main variable name: rain

In [14]:
#Task 2
# DAS (Dataset Attribute Structure) contains metadata
das_url = base_url + ".das"
das_response = requests.get(das_url)
print("Dataset Attributes:")
print(das_response.text[:1000])

Dataset Attributes:
Attributes {
    X {
        String standard_name "longitude";
        Float32 pointwidth 0.5;
        Int32 gridtype 1;
        String units "degree_east";
    }
    T {
        Float32 pointwidth 1.0;
        String calendar "360";
        Int32 gridtype 0;
        String units "months since 1960-01-01";
    }
    Y {
        String standard_name "latitude";
        Float32 pointwidth 0.5;
        Int32 gridtype 0;
        String units "degree_north";
    }
    rain {
        Int32 pointwidth 0;
        String standard_name "lwe_precipitation_rate";
        Float32 file_missing_value -999.0;
        String history "Boxes with less than 0.0% dropped";
        Float32 missing_value NaN;
        String units "mm/day";
        String long_name "Monthly Precipitation";
    }
NC_GLOBAL {
    String Conventions "IRIDL";
}
}



Task 3: What I learned from the DAS output

What does this dataset contain?
This dataset contains monthly precipitation (variable name: rain). The metadata shows long_name = "Monthly Precipitation" and standard_name = "lwe_precipitation_rate".

What time period does it cover?
The time dimension is T, with T = 324 and units "months since 1960-01-01" (calendar "360").
That means it spans 324 months = 27 years, approximately 1960-01 to 1986-12.

What geographic region does it cover?
The dimensions are Y = 360 (latitude) and X = 720 (longitude), with grid spacing pointwidth = 0.5.
This indicates a global 0.5° latitude–longitude grid (lat in degrees north, lon in degrees east).

What are the units of the main variable?
The main variable, rain, has units "mm/day".